# 04 – Profiles

Esplorazione e data cleaning del dataset `profiles.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `username` | Nome utente MAL (chiave primaria) |
| `gender` | Genere dichiarato |
| `birthday` | Data di nascita |
| `location` | Paese dell'utente |
| `joined` | Data di iscrizione a MAL |
| `watching` | Numero di anime attualmente in visione |
| `completed` | Numero di anime completati |
| `on_hold` | Numero di anime in pausa |
| `dropped` | Numero di anime abbandonati |
| `plan_to_watch` | Numero di anime in lista d'attesa |

## 1. Import e caricamento dati

Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [1]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze

df_pr = pd.read_csv('../datasets/profiles.csv')
print(f'Shape: {df_pr.shape}')
print()
df_pr.info()
df_pr.head()

Shape: (337155, 10)

<class 'pandas.DataFrame'>
RangeIndex: 337155 entries, 0 to 337154
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   username       337154 non-null  str  
 1   gender         166279 non-null  str  
 2   birthday       121329 non-null  str  
 3   location       337155 non-null  str  
 4   joined         335479 non-null  str  
 5   watching       335477 non-null  str  
 6   completed      335477 non-null  str  
 7   on_hold        335477 non-null  str  
 8   dropped        335477 non-null  str  
 9   plan_to_watch  335477 non-null  str  
dtypes: str(10)
memory usage: 25.7 MB


,username,gender,birthday,location,joined,watching,completed,on_hold,dropped,plan_to_watch
0,ishikawas,NaN,NaN,South Korea,NaN,NaN,NaN,NaN,NaN,NaN
1,CKK2,NaN,NaN,United States,"Dec 1, 2018",3,182,15,0,405
2,--------788,Female,NaN,Mexico,"Oct 4, 2022",1,64,0,0,1
3,potatoaris,NaN,NaN,Spain,"Oct 2, 2018",5,1,0,0,4
4,Rinrintan,NaN,NaN,Japan,"May 12, 2019",20,311,40,16,34


Il dataset contiene **337.155 righe** e **10 colonne**. I tipi di dati richiedono conversione: le colonne statistiche come `watching`, `completed`, etc sono `str` invece di `int64`, e `birthday` e `joined` sono stringhe invece di `datetime`.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [2]:
n_originale = len(df_pr)

mask_dup = df_pr.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_pr[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_pr.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_pr):,}')

Righe totali coinvolte in duplicazioni : 0
  → prime occorrenze mantenute         : 0
  → occorrenze extra rimosse           : 0

Righe prima della rimozione : 337,155
Righe dopo la rimozione     : 337,155


## 1.2 Rimozione profili vuoti

Verifichiamo la presenza di profili utente senza nessun dato statistico. 

In [3]:
stat_cols = ['watching', 'completed', 'on_hold', 'dropped', 'plan_to_watch']
mask_no_stats = df_pr[stat_cols].isna().all(axis=1)
print(f'Profili senza dati statistici : {mask_no_stats.sum():,}')
print(f'  di cui anche senza joined   : {(mask_no_stats & df_pr["joined"].isna()).sum():,}')
print()
with pd.option_context('display.expand_frame_repr', False, 'display.max_columns', None):
    print('Esempio profili vuoti:')
    print(df_pr[mask_no_stats][['username', 'gender', 'location', 'joined'] + stat_cols].head())
    print()
    print('Esempio profili vuoti con joined:')
    print(df_pr[mask_no_stats & df_pr['joined'].notna()][['username', 'gender', 'location', 'joined'] + stat_cols].head())

Profili senza dati statistici : 1,678
  di cui anche senza joined   : 1,676

Esempio profili vuoti:
       username gender       location joined watching completed on_hold dropped plan_to_watch
0     ishikawas    NaN    South Korea    NaN      NaN       NaN     NaN     NaN           NaN
12  potatobeann    NaN         France    NaN      NaN       NaN     NaN     NaN           NaN
20   potatochib    NaN    South Korea    NaN      NaN       NaN     NaN     NaN           NaN
36     ishixawa    NaN          Japan    NaN      NaN       NaN     NaN     NaN           NaN
71   ---KING---    NaN  United States    NaN      NaN       NaN     NaN     NaN           NaN

Esempio profili vuoti con joined:
                username  gender       location        joined watching completed on_hold dropped plan_to_watch
6352            -Lovely-  Female    South Korea  Sep 10, 2009      NaN       NaN     NaN     NaN           NaN
174904  XxRetribution2xX  Female  United States   Sep 3, 2007      NaN       Na

Dalla verifica risultano 1,678 profili senza dati statistici da cui 1,676 senza la data `joined`. Abbiamo deciso di rimuoverli in quanto non contribuiscono a nessun dato statistico e solo la location in se è un dato poco informativo. 

In [4]:
n_prima = len(df_pr)
df_pr = df_pr[~mask_no_stats].copy()
print(f'Righe prima della rimozione : {n_prima:,}')
print(f'Righe dopo la rimozione     : {len(df_pr):,}')

Righe prima della rimozione : 337,155
Righe dopo la rimozione     : 335,477


## 2. Analisi colonna per colonna

### 2.1 `username`

È la **chiave primaria** del dataset. Deve essere non nulla e univoca. I duplicati non sono attesi.

In [5]:
df_pr['username'] = df_pr['username'].str.strip()
analyze(df_pr['username'])


  Nome serie                     username
  dtype                          str
  Dimensione                     335,477
  Range indice                   1 … 337154
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   335,477
  Valori non nulli               335,476  (100.00%)
  Null / NaN                     1  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   335,476  (100.00%)
  Valori duplicati               0  (0.00%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  16  caratteri
  Lunghezza media                9.33  caratteri
  Lunghezza mediana              9.0000  caratteri
  Dev. standard lunghezza        2.85
  IQR lunghezza                  4.0000

Valori di lun

**Osservazioni:**
- Risulta un valore nullo in `username`. Rimuoviamo la riga in quanti un profilo senza identificatore non è utilizzabile.
- Nessun duplicato. È la chiave primaria del dataset. Nessuna ulteriore pulizia necessaria. 

In [6]:
df_pr.dropna(subset=['username'], inplace=True)
print(f'Righe dopo pulizia username         : {len(df_pr):,}')

Righe dopo pulizia username         : 335,476


### 2.2 `gender`

Colonna categorica con il genere dichiarato dall'utente.

In [7]:
df_pr['gender'] = df_pr['gender'].str.strip()
analyze(df_pr['gender'])


  Nome serie                     gender
  dtype                          str
  Dimensione                     335,476
  Range indice                   1 … 337154
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   335,476
  Valori non nulli               166,277  (49.56%)
  Null / NaN                     169,199  (50.44%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   3  (0.00%)
  Valori duplicati               166,274  (49.56%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  4  caratteri
  Lunghezza max                  10  caratteri
  Lunghezza media                4.65  caratteri
  Lunghezza mediana              4.0000  caratteri
  Dev. standard lunghezza        1.22
  IQR lunghezza                  2.0000

Valori di 

- Si nota la presenza di 169.199 valori nulli (~50%) che non è un errore in quanto l'inserimento del genere su MAL è opzionale. 
- Ci sono 3 valori unici `Male`, `Female`, `Non-Binary` che rispecchiano le opzioni presenti su MAL. 

**Nessuna pulizia necessaria.** 

### 2.3 `birthday`

Data di nascita dell'utente, memorizzata come stringa in formato libero (es. `'Jul 7, 1989'`, `'2003'`, `'Jan 1'`). Invece di convertire a `datetime64` — che causerebbe un'elevata perdita di dati per le date parziali — estraiamo solo l'anno di nascita come intero (`Int64`). Gli anni fuori dal range plausibile `1900–2010` vengono impostati a `null`. I null originali sono strutturali: campo opzionale su MAL.

In [8]:
df_pr['birthday'] = df_pr['birthday'].str.strip()
analyze(df_pr['birthday'])


  Nome serie                     birthday
  dtype                          str
  Dimensione                     335,476
  Range indice                   1 … 337154
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   335,476
  Valori non nulli               121,327  (36.17%)
  Null / NaN                     214,149  (63.83%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   11,344  (3.38%)
  Valori duplicati               109,983  (32.78%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  12  caratteri
  Lunghezza media                9.90  caratteri
  Lunghezza mediana              11.0000  caratteri
  Dev. standard lunghezza        2.88
  IQR lunghezza                  6.0000

Va

**Pulizia necessaria:**
- Estraiamo il primo numero a 4 cifre trovato nella stringa come anno di nascita (`\b\d{4}\b`): cattura `'1989'` da `'Jul 7, 1989'` e `'1989'` da `'1989'`, ma restituisce `null` per `'Jan 1'` (nessun anno)
- Anni fuori dal range `1900–2010` → `null` (non plausibili per un utente MAL)
- I null originali (~36%) sono strutturali

In [9]:
n_null_before   = df_pr['birthday'].isna().sum()
n_nonnull_before = df_pr['birthday'].notna().sum()

# Estrai anno a 4 cifre
df_pr['birthday'] = df_pr['birthday'].str.extract(r'(\b\d{4}\b)')[0].astype('Int64')

n_with_year = df_pr['birthday'].notna().sum()
n_no_year   = n_nonnull_before - n_with_year  # es. "Jan 1" → nessun anno

# Rimuovi anni fuori range plausibile
mask_invalid = df_pr['birthday'].notna() & ~df_pr['birthday'].between(1900, 2010)
n_invalid = mask_invalid.sum()
df_pr.loc[mask_invalid, 'birthday'] = pd.NA

n_null_after = df_pr['birthday'].isna().sum()

print(f'Valori non-null prima della conversione : {n_nonnull_before:,}')
print(f'  → anni estratti (1900–2010)           : {n_with_year - n_invalid:,}')
print(f'  → anni fuori range → null             : {n_invalid:,}')
print(f'  → senza anno (es. "Jan 1") → null     : {n_no_year:,}')
print(f'Null totali dopo la conversione         : {n_null_after:,} ({n_null_after / len(df_pr) * 100:.1f}%)')
print(f'birthday dtype                          : {df_pr["birthday"].dtype}')
print(f'Range                                   : {df_pr["birthday"].min()} → {df_pr["birthday"].max()}')
print()
df_pr[['username', 'birthday']].dropna(subset=['birthday']).head(10)

Valori non-null prima della conversione : 121,327
  → anni estratti (1900–2010)           : 93,509
  → anni fuori range → null             : 484
  → senza anno (es. "Jan 1") → null     : 27,334
Null totali dopo la conversione         : 241,967 (72.1%)
birthday dtype                          : Int64
Range                                   : 1900 → 2010



,username,birthday
15,RinsAl,1989
19,ishimori,2000
23,Karinnn___,2001
25,Rins_,1988
29,FollowValen,1996
30,ishiruchan,2003
35,----Blank----,2001
41,CKRpl,2002
46,Rinshuu,1997
53,----phoebelyn,1994


### 2.4 `location`

Paese dell'utente.

In [10]:
df_pr['location'] = df_pr['location'].str.strip()
analyze(df_pr['location'])


  Nome serie                     location
  dtype                          str
  Dimensione                     335,476
  Range indice                   1 … 337154
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   335,476
  Valori non nulli               335,476  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   22  (0.01%)
  Valori duplicati               335,454  (99.99%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  5  caratteri
  Lunghezza max                  14  caratteri
  Lunghezza media                8.03  caratteri
  Lunghezza mediana              7.0000  caratteri
  Dev. standard lunghezza        3.39
  IQR lunghezza                  8.0000

Valori di lun

Nessun null. La colonna presenta esattamente 22 valori distinti (paesi). 
**Nessuna pulizia necessaria.**  


In [11]:
print(f'Null in location  : {df_pr["location"].isna().sum()}')
print(f'Valori unici      : {df_pr["location"].nunique()}')
print()
print('Distribuzione location:')
print(df_pr['location'].value_counts())

Null in location  : 0
Valori unici      : 22

Distribuzione location:
location
Japan             97820
United States     64930
Germany           26226
United Kingdom    19433
Thailand          16332
Argentina         13165
China             13128
Spain             12837
France             9983
Australia          9651
Mexico             6597
South Korea        6490
Turkey             6485
Italy              6440
Indonesia          3314
Brazil             3279
Vietnam            3257
South Africa       3253
Philippines        3240
Egypt              3222
India              3207
Canada             3187
Name: count, dtype: int64


### 2.5 `joined`

Data di iscrizione a MAL, memorizzata come stringa. Richiede conversione a `datetime64`.

In [12]:
df_pr['joined'] = df_pr['joined'].str.strip()
analyze(df_pr['joined'])


  Nome serie                     joined
  dtype                          str
  Dimensione                     335,476
  Range indice                   1 … 337154
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   335,476
  Valori non nulli               335,476  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   6,802  (2.03%)
  Valori duplicati               328,674  (97.97%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  11  caratteri
  Lunghezza max                  12  caratteri
  Lunghezza media                11.71  caratteri
  Lunghezza mediana              12.0000  caratteri
  Dev. standard lunghezza        0.46
  IQR lunghezza                  1.0000

Valori di

**Osservazioni:**  
- Non ci sono valori nulli. 
- La lunghezza è consistente: 11 - 12 caratteri che dimostra mancanza di anomalie 
- I valori sono stringhe in formato `'Dec 1, 2018'` che vanno convertiti a `datetime64`.  

**Nessuna pulizia necessaria**

In [13]:
df_pr['joined'] = pd.to_datetime(df_pr['joined'], errors='coerce')
print(f'joined dtype  : {df_pr["joined"].dtype}')
print(f'Null residui  : {df_pr["joined"].isna().sum()}')
print(f'Range         : {df_pr["joined"].min()} → {df_pr["joined"].max()}')

joined dtype  : datetime64[us]
Null residui  : 0
Range         : 2004-11-05 00:00:00 → 2025-11-02 00:00:00


### 2.6 `watching`

Numero di anime attualmente in visione.

In [14]:
df_pr['watching'] = df_pr['watching'].str.strip()
analyze(df_pr['watching'])


  Nome serie                     watching
  dtype                          str
  Dimensione                     335,476
  Range indice                   1 … 337154
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   335,476
  Valori non nulli               335,476  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   942  (0.28%)
  Valori duplicati               334,534  (99.72%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  6  caratteri
  Lunghezza media                1.42  caratteri
  Lunghezza mediana              1.0000  caratteri
  Dev. standard lunghezza        0.55
  IQR lunghezza                  1.0000

Valori di lun

**Osservazioni:**
- Non ci sono valori nulli.
- 335,290 su 335,476 contengono solo numeri. Questo perchè il resto delle celle possono contenere separatore come la virgola. Per assicurarci che non ci sono altri caratteri, facciamo un controllo prima della pulizia. 
- Effetuiamo la rimozione della virgola e facciamo la conversione da `str` a `Int64`.

In [15]:
mask_anomali = df_pr['watching'].notna() & df_pr['watching'].str.contains(r'[^\d,]', na=False)
print(f'Valori con caratteri inattesi: {mask_anomali.sum()}')
if mask_anomali.any():
    print(df_pr.loc[mask_anomali, 'watching'].unique()[:10])
print()

mask_virgola = df_pr['watching'].str.contains(',', na=False)

n_null_before = df_pr['watching'].isna().sum()
df_pr['watching'] = pd.to_numeric(df_pr['watching'].str.replace(',', '', regex=False), errors='coerce').astype('Int64')
n_null_after = df_pr['watching'].isna().sum()
print(f'watching dtype    : {df_pr["watching"].dtype}')
print()
df_pr.loc[mask_virgola, ['username', 'watching']].head(10)

Valori con caratteri inattesi: 0

watching dtype    : Int64



,username,watching
952,KarnKumar,2617
2019,-Aya_Chan-,2055
3428,Fovexes71125,1930
4333,FoxyRubyRouge,2216
4458,Cafer,1041
7258,izumi-kun98,1023
8880,KawaseGawa,1332
8938,-Sabre-,2772
9040,purplepinapples,2587
9315,Cameo-Taku,1747


### 2.7 `completed`

Numero di anime completati.

In [16]:
df_pr['completed'] = df_pr['completed'].str.strip()
analyze(df_pr['completed'])


  Nome serie                     completed
  dtype                          str
  Dimensione                     335,476
  Range indice                   1 … 337154
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   335,476
  Valori non nulli               335,476  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   3,380  (1.01%)
  Valori duplicati               332,096  (98.99%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  6  caratteri
  Lunghezza media                2.64  caratteri
  Lunghezza mediana              3.0000  caratteri
  Dev. standard lunghezza        0.79
  IQR lunghezza                  1.0000

Valori di 

**Osservazioni:**
- Non ci sono valori nulli.
- 323,357 su 335,476 contengono solo numeri. Questo perchè il resto delle celle possono contenere separatore come la virgola. Per assicurarci che non ci sono altri caratteri, facciamo un controllo prima della pulizia. 
- Effetuiamo la rimozione della virgola e facciamo la conversione da `str` a `Int64`.

In [17]:
mask_anomali = df_pr['completed'].notna() & df_pr['completed'].str.contains(r'[^\d,]', na=False)
print(f'Valori con caratteri inattesi: {mask_anomali.sum()}')
if mask_anomali.any():
    print(df_pr.loc[mask_anomali, 'completed'].unique()[:10])
print()

mask_virgola = df_pr['completed'].str.contains(',', na=False)

n_null_before = df_pr['completed'].isna().sum()
df_pr['completed'] = pd.to_numeric(df_pr['completed'].str.replace(',', '', regex=False), errors='coerce').astype('Int64')
n_null_after = df_pr['completed'].isna().sum()
print(f'completed dtype   : {df_pr["completed"].dtype}')
print(f'Null prima → dopo : {n_null_before} → {n_null_after}  (nuovi null = {n_null_after - n_null_before})')
print(f'Range             : {df_pr["completed"].min()} → {df_pr["completed"].max()}')
print()
df_pr.loc[mask_virgola, ['username', 'completed']].head(10)

Valori con caratteri inattesi: 0

completed dtype   : Int64
Null prima → dopo : 0 → 0  (nuovi null = 0)
Range             : 0 → 28313



,username,completed
5,Karincakes,2899
48,Karinyia,2028
58,----shia,2581
164,Karkato,1003
194,Rinxx,1635
219,ThomasGremory,1807
271,Foms,2092
352,RioAl,1439
365,CLYDESDALE,1018
392,isokana,1427


### 2.8 `on_hold`

Numero di anime in pausa. 

In [18]:
df_pr['on_hold'] = df_pr['on_hold'].str.strip()
analyze(df_pr['on_hold'])


  Nome serie                     on_hold
  dtype                          str
  Dimensione                     335,476
  Range indice                   1 … 337154
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   335,476
  Valori non nulli               335,476  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   738  (0.22%)
  Valori duplicati               334,738  (99.78%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  6  caratteri
  Lunghezza media                1.28  caratteri
  Lunghezza mediana              1.0000  caratteri
  Dev. standard lunghezza        0.49
  IQR lunghezza                  1.0000

Valori di lung

**Osservazioni:**
- Non ci sono valori nulli.
- 335,382 su 335,476 contengono solo numeri. Questo perchè il resto delle celle possono contenere separatore come la virgola. Per assicurarci che non ci sono altri caratteri, facciamo un controllo prima della pulizia. 
- Effetuiamo la rimozione della virgola e facciamo la conversione da `str` a `Int64`.

In [19]:
mask_anomali = df_pr['on_hold'].notna() & df_pr['on_hold'].str.contains(r'[^\d,]', na=False)
print(f'Valori con caratteri inattesi: {mask_anomali.sum()}')
if mask_anomali.any():
    print(df_pr.loc[mask_anomali, 'on_hold'].unique()[:10])
print()

mask_virgola = df_pr['on_hold'].str.contains(',', na=False)

n_null_before = df_pr['on_hold'].isna().sum()
df_pr['on_hold'] = pd.to_numeric(df_pr['on_hold'].str.replace(',', '', regex=False), errors='coerce').astype('Int64')
n_null_after = df_pr['on_hold'].isna().sum()
print(f'on_hold dtype     : {df_pr["on_hold"].dtype}')
print(f'Null prima → dopo : {n_null_before} → {n_null_after}  (nuovi null = {n_null_after - n_null_before})')
print(f'Range             : {df_pr["on_hold"].min()} → {df_pr["on_hold"].max()}')
print()
df_pr.loc[mask_virgola, ['username', 'on_hold']].head(10)

Valori con caratteri inattesi: 0

on_hold dtype     : Int64
Null prima → dopo : 0 → 0  (nuovi null = 0)
Range             : 0 → 18403



,username,on_hold
5051,-Joelle-,1492
6149,-Lazyan-,1660
7010,-Mooners-,1044
13366,avaster23,1014
15898,axios1331,2097
29865,bbblgmxbsh,1151
31137,be_lost_sakura,1619
35316,jtcurrey,1261
35321,bence0518,1517
37318,Ryukiation,1130


### 2.9 `dropped`

Numero di anime abbandonati. 

In [20]:
df_pr['dropped'] = df_pr['dropped'].str.strip()
analyze(df_pr['dropped'])


  Nome serie                     dropped
  dtype                          str
  Dimensione                     335,476
  Range indice                   1 … 337154
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   335,476
  Valori non nulli               335,476  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   900  (0.27%)
  Valori duplicati               334,576  (99.73%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  6  caratteri
  Lunghezza media                1.34  caratteri
  Lunghezza mediana              1.0000  caratteri
  Dev. standard lunghezza        0.53
  IQR lunghezza                  1.0000

Valori di lung

**Osservazioni:**
- Non ci sono valori nulli.
- 335,292 su 335,476 contengono solo numeri. Questo perchè il resto delle celle possono contenere separatore come la virgola. Per assicurarci che non ci sono altri caratteri, facciamo un controllo prima della pulizia. 
- Effetuiamo la rimozione della virgola e facciamo la conversione da `str` a `Int64`.

In [21]:
mask_anomali = df_pr['dropped'].notna() & df_pr['dropped'].str.contains(r'[^\d,]', na=False)
print(f'Valori con caratteri inattesi: {mask_anomali.sum()}')
if mask_anomali.any():
    print(df_pr.loc[mask_anomali, 'dropped'].unique()[:10])
print()

mask_virgola = df_pr['dropped'].str.contains(',', na=False)

n_null_before = df_pr['dropped'].isna().sum()
df_pr['dropped'] = pd.to_numeric(df_pr['dropped'].str.replace(',', '', regex=False), errors='coerce').astype('Int64')
n_null_after = df_pr['dropped'].isna().sum()
print(f'dropped dtype     : {df_pr["dropped"].dtype}')
print()
df_pr.loc[mask_virgola, ['username', 'dropped']].head(10)

Valori con caratteri inattesi: 0

dropped dtype     : Int64



,username,dropped
4759,itzgherrinee,9862
7113,Calicolex,1635
9040,purplepinapples,19601
10825,Cancerous_Bones,2063
11405,MotherThe,1380
15898,axios1331,1097
20001,Rosaliae,1279
21715,baby_minu,1014
22611,Rowinftw,1089
26154,banba_84,1315


### 2.10 `plan_to_watch`

Numero di anime in lista d'attesa. 

In [22]:
df_pr['plan_to_watch'] = df_pr['plan_to_watch'].str.strip()
analyze(df_pr['plan_to_watch'])


  Nome serie                     plan_to_watch
  dtype                          str
  Dimensione                     335,476
  Range indice                   1 … 337154
  Tipo                           STRINGA / OGGETTO

Conteggi di base
────────────────────────────────────────────────────────────────────────────────

  Righe totali                   335,476
  Valori non nulli               335,476  (100.00%)
  Null / NaN                     0  (0.00%)
    di cui stringhe vuote        0  (convertite a NULL)
  Valori unici                   2,538  (0.76%)
  Valori duplicati               332,938  (99.24%)

Analisi lunghezza stringhe
────────────────────────────────────────────────────────────────────────────────

  Lunghezza min                  1  caratteri
  Lunghezza max                  6  caratteri
  Lunghezza media                2.03  caratteri
  Lunghezza mediana              2.0000  caratteri
  Dev. standard lunghezza        0.82
  IQR lunghezza                  2.0000

Valori

**Osservazioni:**
- Non ci sono valori nulli.
- 331,792 su 335,476 contengono solo numeri. Questo perchè il resto delle celle possono contenere separatore come la virgola. Per assicurarci che non ci sono altri caratteri, facciamo un controllo prima della pulizia. 
- Effetuiamo la rimozione della virgola e facciamo la conversione da `str` a `Int64`.

In [23]:
mask_anomali = df_pr['plan_to_watch'].notna() & df_pr['plan_to_watch'].str.contains(r'[^\d,]', na=False)
print(f'Valori con caratteri inattesi: {mask_anomali.sum()}')
if mask_anomali.any():
    print(df_pr.loc[mask_anomali, 'plan_to_watch'].unique()[:10])
print()

mask_virgola = df_pr['plan_to_watch'].str.contains(',', na=False)

n_null_before = df_pr['plan_to_watch'].isna().sum()
df_pr['plan_to_watch'] = pd.to_numeric(df_pr['plan_to_watch'].str.replace(',', '', regex=False), errors='coerce').astype('Int64')
n_null_after = df_pr['plan_to_watch'].isna().sum()
print(f'plan_to_watch dtype : {df_pr["plan_to_watch"].dtype}')
print()
df_pr.loc[mask_virgola, ['username', 'plan_to_watch']].head(10)

Valori con caratteri inattesi: 0

plan_to_watch dtype : Int64



,username,plan_to_watch
17,Thoma3938,6289
108,Rinthian,1030
219,ThomasGremory,1004
454,Thomas_Atwood,1794
507,Karma,4026
663,isshuukan,2014
921,-Abhishek-,1427
953,FootSlut,2488
1370,Foreall908,1099
1507,Thotank,1406


### 2.11 Chiave primaria `username`

Verifichiamo che `username` sia univoco dopo tutte le operazioni di pulizia.

In [24]:
n_pk_dup = df_pr.duplicated(subset=['username'], keep=False).sum()
print(f'Duplicati su username: {n_pk_dup}')

if n_pk_dup > 0:
    print('→ Rimozione duplicati sulla chiave primaria...')
    df_pr.drop_duplicates(subset=['username'], keep='first', inplace=True)
    print(f'Righe dopo la rimozione: {len(df_pr):,}')
else:
    print('→ Chiave primaria già univoca, nessuna operazione richiesta.')

Duplicati su username: 0
→ Chiave primaria già univoca, nessuna operazione richiesta.


## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effettuiamo il salvataggio del dataset finale.

In [25]:
print('Riepilogo Dataset Pulito')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_pr):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_pr):>10,}')
print()

df_pr.to_csv('../datasets_cleaned/profiles_clean.csv', index=False)
print('Salvato: datasets_cleaned/profiles_clean.csv')

Riepilogo Dataset Pulito
Righe originali      :    337,155
Righe dopo cleaning  :    335,476
Righe rimosse totali :      1,679

Salvato: datasets_cleaned/profiles_clean.csv
